In [1]:
from aiida.engine import run_get_node
from topworkchain import PrototypeTopWorkChain
from aiida.orm import FolderData
from aiida import load_profile
load_profile()

Profile<uuid='8982fbed0baf4361bcf833f5b70e51c4' name='presto'>

In [8]:
#Run the model
results, node = run_get_node(PrototypeTopWorkChain)
#Load the FolderData node containing the compressed cube files (a folder of files stored on the AiiDA database.)
cube_folder = results["cube_folder"]

uuid: 1fa57720-6a7c-4878-8e08-a1076974c12d (pk: 1950) (subworkchains.OrcaWorkChain)
uuid: ccef92a0-1c1f-482f-ba5f-590d372b4f86 (pk: 1958)
uuid: 9236fe10-ec61-4883-8a54-4a7397c8e6f2 (pk: 1973)
uuid: 7b72966d-3dc8-446b-a29c-85bd856bd38f (pk: 1987)
uuid: c94194ee-1253-4c6c-a35d-d3cc936b7b78 (pk: 2002)
uuid: d7bc0891-d083-45cf-b0da-fc1677630177 (pk: 2016)
uuid: 43870bd5-a9c0-4606-bd00-4e36a700649e (pk: 2031)
uuid: daf740a5-60a9-41bf-8300-a8130cfaeda6 (pk: 2045)
uuid: 822bd3fd-3e17-4d47-8867-cc35d902f2f2 (pk: 2059)
uuid: fce05f11-fe91-4725-821e-d2ca65879d31 (pk: 2073)
uuid: 3131467d-8d03-4268-bce7-4961a360a6b2 (pk: 2088)
uuid: 010c5e66-2a82-4110-87d0-05c058164f43 (pk: 2102)
uuid: f0363c87-f531-4d9b-8ca7-9adbf0b107e6 (pk: 2117)
uuid: 9997ff3b-5365-4050-bb94-ff5a293ec2ad (pk: 2131)
uuid: 6ffa8afe-ff43-40d7-920e-f1dee61be4ce (pk: 2146)
uuid: 9753ff14-36b4-4d68-90c6-2adadb772139 (pk: 2160)
uuid: d8b8afb0-8257-4d18-9dfe-c7edb8e58de5 (pk: 2174)
uuid: 2ab6fc38-f632-4b7e-8825-79fdb34fb897 (pk: 2188

In [9]:
import nglview as nv
import ipywidgets as widgets
import traitlets as tl
import ase.io
import io
import os
from ase.build import molecule
import tempfile
from pathlib import Path
from ipywidgets import Layout

In [11]:
# Create an output directory for the temporary cube files
output_dir = Path("extracted_cubes")
output_dir.mkdir(exist_ok=True)

# Create a list of the cube files in the data node
names = cube_folder.list_object_names()

# Read in the cube files and write them to temporary files
for name in names:
    with cube_folder.open(name, mode="rb") as file_in:
        with open(f"{output_dir}/{name}.cube", "wb") as file_out:
            file_out.write(file_in.read())

cube_names = sorted([name for name in os.listdir("./extracted_cubes") if name.endswith(".cube")])

# Create a list of the paths to the temporary cube files
CUBE_PATHS = [f"{output_dir}/{name}.cube" for name in names]

# Create dropdown widgets for selecting the cube files
cube_file_1 = widgets.Dropdown(
    options=cube_names,
    value=cube_names[0] if cube_names else None,
    description="Molecule 1"
 )

cube_file_2 = widgets.Dropdown(
    options=cube_names,
    value=cube_names[1] if len(cube_names) > 1 else (cube_names[0] if cube_names else None),
    description="Molecule 2"
 )

# Initialize views with current dropdown selections
CUBE_PATH_1 = f"./extracted_cubes/{cube_file_1.value}"
CUBE_PATH_2 = f"./extracted_cubes/{cube_file_2.value}"

# Create NGL views for the selected cube files
molecule1 = nv.show_ase(ase.io.read(CUBE_PATH_1))
molecule2 = nv.show_ase(ase.io.read(CUBE_PATH_2))
NTO_1 = molecule1.add_component(CUBE_PATH_1, ext="cube", default_representation=False)
NTO_2 = molecule2.add_component(CUBE_PATH_2, ext="cube", default_representation=False)


# Create sliders for adjusting the isovalue (not visible in the NGL view, but used to set the isovalue of the surfaces)
isovalue = 0.001
# positive_level = widgets.FloatLogSlider(
#     value=1e-3,
#     base=10,
#     min=-5,
#     max=-1,
#     step=0.1,
#     description="+ isovalue",
#     readout_format=".1e",
#  )
# negative_level = widgets.FloatLogSlider(
#     value=1e-3,
#     base=10,
#     min=-5,
#     max=-1,
#     step=0.1,
#     description="- isovalue",
#     readout_format=".1e",
#  )

# Create a dropdown for selecting the colour scheme of the surfaces
colour_scheme = widgets.Dropdown(
    options=["default", "electrostatic", "element"],
    value="default",
    description="Colour scheme",
 )

def get_surface_colors(scheme):
    if scheme == "default":
        return "#3366ff", "#ff4d4d"
    if scheme == "electrostatic":
        return "#f713d5", "#3366ff"
    if scheme == "element":
        return "#7a4dff", "#eeff00"
    return "#3366ff", "#ff4d4d"

# Set the initial surface colors based on the default colour scheme
positive_color, negative_color = get_surface_colors(colour_scheme.value)
status = widgets.HTML()

# Define functions to redraw the surfaces when the sliders or colour scheme are changed
def redraw_NTO_1(_=None):
    NTO_1.clear()
    NTO_1.add_surface(
        color=positive_color,
        isolevelType="value",
        isolevel=float(isovalue),
        opacity=0.5,
    )
    NTO_1.add_surface(
        color=negative_color,
        isolevelType="value",
        isolevel=-float(isovalue),
        opacity=0.5,
    )
def redraw_NTO_2(_=None):
    NTO_2.clear()
    NTO_2.add_surface(
        color=positive_color,
        isolevelType="value",
        isolevel=float(isovalue),
        opacity=0.5,
    )
    NTO_2.add_surface(
        color=negative_color,
        isolevelType="value",
        isolevel=-float(isovalue),
        opacity=0.5,
    )
def redraw_surfaces(_=None):
    global positive_color, negative_color
    positive_color, negative_color = get_surface_colors(colour_scheme.value)
    redraw_NTO_1()
    redraw_NTO_2()

def update_NTO_1(_=None):
    global NTO_1
    NTO_1.clear()
    NTO_1 = molecule1.add_component(f"./extracted_cubes/{cube_file_1.value}", ext="cube", default_representation=False)  # Add path!
    redraw_NTO_1()
    
def update_NTO_2(_=None):
    global NTO_2
    NTO_2.clear()
    NTO_2 = molecule2.add_component(f"./extracted_cubes/{cube_file_2.value}", ext="cube", default_representation=False)  # Add path!
    redraw_NTO_2()

# Set up observers to redraw the surfaces when the sliders or colour scheme are changed
for control in [colour_scheme]:
    control.observe(redraw_surfaces, names="value")

cube_file_1.observe(update_NTO_1, names="value")
cube_file_2.observe(update_NTO_2, names="value")

redraw_surfaces()

# Arrange the widgets in a layout
controls = widgets.VBox(
    [
        widgets.HTML("<b>Cube isosurfaces</b>"),
        widgets.VBox([colour_scheme], layout = Layout(border="1px solid black", width="40%")),
        status,
    ],
 )
molecule1_box = widgets.VBox([cube_file_1, molecule1], layout= Layout(width="100%", flex="1 1 50%", min_width="0"))
molecule2_box = widgets.VBox([cube_file_2, molecule2], layout= Layout(width="100%", flex="1 1 50%", min_width="0"))
molecule_box = widgets.HBox([molecule1_box, molecule2_box], layout=Layout(width="100%", flex="1 1 100%"))

widgets.VBox([controls, molecule_box], layout = Layout(width="100%", flex="1 1 100%"))


In [7]:
cube_names

['acrolein.s1.mo14a.cube',
 'acrolein.s1.mo15a.cube',
 'compressed_cube.cube',
 'final_test.cube',
 's1.14.cube',
 's1.15.cube',
 's10.14.cube',
 's10.15.cube',
 's11.14.cube',
 's11.15.cube',
 's12.14.cube',
 's12.15.cube',
 's13.12.cube',
 's13.13.cube',
 's13.14.cube',
 's13.15.cube',
 's13.16.cube',
 's13.17.cube',
 's14.13.cube',
 's14.14.cube',
 's14.15.cube',
 's14.16.cube',
 's15.14.cube',
 's15.15.cube',
 's16.12.cube',
 's16.13.cube',
 's16.14.cube',
 's16.15.cube',
 's16.16.cube',
 's16.17.cube',
 's17.13.cube',
 's17.14.cube',
 's17.15.cube',
 's17.16.cube',
 's18.13.cube',
 's18.14.cube',
 's18.15.cube',
 's18.16.cube',
 's19.14.cube',
 's19.15.cube',
 's2.14.cube',
 's2.15.cube',
 's20.13.cube',
 's20.14.cube',
 's20.15.cube',
 's20.16.cube',
 's21.14.cube',
 's21.15.cube',
 's22.14.cube',
 's22.15.cube',
 's23.13.cube',
 's23.14.cube',
 's23.15.cube',
 's23.16.cube',
 's24.14.cube',
 's24.15.cube',
 's25.11.cube',
 's25.12.cube',
 's25.13.cube',
 's25.14.cube',
 's25.15.

In [6]:
test = nv.show_ase(ase.io.read("./extracted_cubes/s6.14.cube"))
test.add_component("./extracted_cubes/s6.14.cube", ext="cube", default_representation=False)

test.display()

NGLWidget()